# 技能5 · Day 6 上机：IMRaD 论文写作 -- 用 Python 拆解真实论文 + 撰写营销研究各部分

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实论文/库）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **arxiv** Python 包下载/解析真实论文，自动提取其 IMRaD 各部分结构
2. 撰写 **Introduction**（漏斗结构：背景 -> 问题 -> 空白 -> 贡献 -> 结构），基于一个营销研究问题
3. 撰写 **Methods**（研究设计/数据收集/分析方法），确保可复现性
4. 用 **statsmodels** 跑统计检验（t 检验/卡方/Cohen's d），把结果写成 APA 格式学术表述
5. 撰写 **Discussion**（发现解读/局限性/未来方向），展示对研究的深度理解
6. 生成 **APA 第 7 版**参考文献列表（真实引用，格式准确）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：arxiv（lukasschwab/arxiv.py，1.5k★，arXiv API 封装）+ statsmodels（统计检验）。
营销映射：撰写一篇"营销Agent vs 人工策略效果对比"的 IMRaD 论文。


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ arxiv 包需要网络连接访问 arXiv API。statsmodels/scipy/numpy 离线可用。
> 若无法联网，TODO1 的 solution 提供了预置的论文元数据 fallback。


In [ ]:
# !pip install arxiv statsmodels scipy numpy -q
# arxiv 包需要网络连接；statsmodels/scipy/numpy 离线可用

## 1. 数据集背景与营销映射

**研究主题**：营销Agent vs 人工策略的效果对比（A/B 测试 + 用户访谈）

**真实论文范例**（用于 IMRaD 结构拆解）：

| 论文 | arXiv ID | 用途 |
|------|----------|------|
| ReAct: Synergizing Reasoning and Acting in Language Models | 2210.03629 | Agent 论文的 IMRaD 结构范例（Yao et al., NeurIPS 2022） |
| Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena | 2306.05685 | 评估方法论引用（Zheng et al., NeurIPS 2023） |

**营销 A/B 测试数据**（用于 Results 部分统计分析）：

| 指标 | 对照组（人工） | 实验组（Agent） | 数据来源 |
|------|:-----------:|:-----------:|---------|
| 内容产出效率（篇/天） | ~8 | ~32 | 模拟真实营销团队2个月数据 |
| 内容 CTR（%） | ~2.1 | ~2.8 | 基于行业基准 |
| 用户满意度（1-10） | ~7.2 | ~7.8 | 模拟用户评分 |

> 💡 在真实研究中，这些数据来自你的 A/B 测试平台。本上机用基于行业基准的可复现数据（固定随机种子）。


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from scipy import stats as sp_stats
from statsmodels.stats.weightstats import ttest_ind

print("依赖导入完成：numpy, scipy, statsmodels")
print("arxiv 包将在 TODO1 中按需导入")

## TODO 1：用 arxiv 包下载/解析真实论文，提取 IMRaD 结构

**目标**：用 `arxiv` Python 包获取 ReAct 论文（arXiv 2210.03629）的元数据，解析其摘要中的 IMRaD 结构。

**IMRaD 结构识别方法**：
- Introduction 句子：通常包含背景、动机、问题陈述（"we propose"、"this paper"）
- Methods 句子：描述方法、数据集、实验设计（"we evaluate"、"our approach"）
- Results 句子：报告发现、数据、性能（"we find"、"results show"、"achieves"）
- Discussion/Conclusion 句子：总结贡献、局限性（"we demonstrate"、"in summary"）


In [ ]:
# TODO 1：用 arxiv 包下载/解析真实论文，提取 IMRaD 结构
# 提示：
#   import arxiv
#   client = arxiv.Client()
#   search = arxiv.Search(id_list=["2210.03629"])
#   paper = next(client.results(search))
#   paper.title / paper.authors / paper.summary (摘要)
# 然后解析摘要中的句子，按 IMRaD 结构分类：
#   - Introduction: 包含 "we propose"/"this paper"/"we study" 等
#   - Methods: 包含 "we evaluate"/"our approach"/"we use" 等
#   - Results: 包含 "results show"/"achieves"/"we find" 等
#   - Discussion: 包含 "we demonstrate"/"in summary"/"limitations" 等
# 要求：获取 ReAct 论文元数据，解析摘要的 IMRaD 结构，打印分析结果

# ===== 你的代码 =====
paper_info = None  # TODO: 用 arxiv 包获取 ReAct 论文元数据
imrad_analysis = None  # TODO: 解析摘要的 IMRaD 结构
# ====================

if paper_info:
    print(f"论文标题: {paper_info.get('title', 'N/A')}")
    print(f"作者: {', '.join(paper_info.get('authors', []))}")
    print(f"\nIMRaD 结构分析:")
    for section, sentences in imrad_analysis.items():
        print(f"  {section}: {len(sentences)} 句")


## 2. IMRaD 详解：Introduction（漏斗结构）

Introduction 遵循"漏斗结构"--从大到小，从宽到窄：

```
领域背景（宽）  "AI原生营销正在重塑企业增长方式..."
    |
具体问题（窄）  "但现有营销Agent系统缺乏有效的效果评估方法论..."
    |
研究空白（更窄）  "目前没有研究系统地探讨多Agent营销系统的评估指标体系..."
    |
本文贡献（最窄）  "本文提出一个基于LangGraph的多Agent营销系统架构..."
    |
论文结构  "本文第2节介绍方法，第3节呈现结果，第4节讨论..."
```

**核心要点**：
1. 研究背景：2-3 段，引用行业报告和学术文献
2. 研究问题：1-2 段，明确指出当前系统的问题
3. 研究空白：1 段，通过文献综述指出前人没做什么
4. 本文贡献：3-4 个 bullet points，清晰声明本文做了什么
5. 论文结构：1 段，概述后续各节内容

> 详见独立教材 § 3.6.2。


## TODO 2：撰写 Introduction

**研究主题**：营销Agent vs 人工策略的效果对比

**要求**：
1. 研究背景：AI原生营销发展趋势（引用 McKinsey 2025 报告风格）
2. 研究问题：现有营销 Agent 系统缺乏标准化效果评估
3. 研究空白：尚无研究系统对比 Agent 与人工在营销内容创作上的效果
4. 本文贡献：3-4 个 bullet points
5. 论文结构：概述后续各节

将撰写的 Introduction 存为 `introduction_text` 变量（多行字符串）。


In [ ]:
# TODO 2：撰写 Introduction
# 提示：遵循漏斗结构（背景 -> 问题 -> 空白 -> 贡献 -> 结构）
# 要求：基于"营销Agent vs 人工策略效果对比"研究主题撰写

# ===== 你的代码 =====
introduction_text = None  # TODO: 撰写完整的 Introduction（多行字符串）
# ====================

print(introduction_text)


## 3. IMRaD 详解：Methods（可复现性）

Methods 的核心要求是**可复现性**：别人读完你的 Methods，应该能用同样的方法重复你的研究。

**Methods 五要素**：
1. 研究设计：Design Science Research (DSR) / 混合方法 / 纯定量
2. 系统架构：Agent 系统的架构描述（附图）
3. 数据收集：定量数据（A/B 测试）+ 定性数据（访谈）
4. 评估指标：任务完成率/工具准确率/幻觉率/延迟/成本
5. 数据分析方法：t 检验/Cohen's d/主题分析法

> 详见独立教材 § 3.6.3。


## TODO 3：撰写 Methods

**要求**：
1. 研究设计：采用混合方法设计（定量 A/B 测试 + 定性用户访谈）
2. 数据收集：A/B 测试（N=400，对照组人工 vs 实验组 Agent，2个月）+ 8位营销人员访谈
3. 评估指标：内容产出效率/CTR/用户满意度
4. 数据分析方法：独立样本 t 检验 + Cohen's d 效应量 + 主题分析法（Braun & Clarke, 2006）

将撰写的 Methods 存为 `methods_text` 变量（多行字符串）。


In [ ]:
# TODO 3：撰写 Methods
# 提示：确保可复现性（设计/数据收集/评估指标/分析方法）
# 要求：混合方法设计（A/B测试 + 访谈），t检验 + Cohen's d + 主题分析

# ===== 你的代码 =====
methods_text = None  # TODO: 撰写完整的 Methods（多行字符串）
# ====================

print(methods_text)


## 4. IMRaD 详解：Results（数据说话）

Results 的原则是**先描述再解释**。描述你发现了什么，解释放在 Discussion。

**Results 写作要点**：
1. 用表格呈现核心数据（附表标题和标注）
2. 统计检验结果用 APA 格式：t(df) = X.XX, p < .001, d = X.XX
3. 效应量解读：d=0.2 小，d=0.5 中，d=0.8 大（Cohen, 1988）
4. 先描述定量结果，再描述定性结果

**APA 统计报告格式**：
- t 检验：`实验组的内容产出效率显著高于对照组（t(398) = 25.43, p < .001, d = 2.34）`
- 卡方检验：`转化率差异显著（χ²(1, N=400) = 8.32, p < .01, φ = 0.14）`

> 详见独立教材 § 3.6.4。


## TODO 4：撰写 Results（用 statsmodels 跑统计检验）

**要求**：
1. 生成营销 A/B 测试数据（固定随机种子，N=200/组）
   - 对照组（人工）：内容产出效率均值~8，CTR~2.1%，满意度~7.2
   - 实验组（Agent）：内容产出效率均值~32，CTR~2.8%，满意度~7.8
2. 用 `statsmodels.stats.weightstats.ttest_ind` 跑独立样本 t 检验
3. 计算 Cohen's d 效应量
4. 用 `scipy.stats.chi2_contingency` 跑卡方检验（CTR 是否显著提升）
5. 将统计结果写成 APA 格式学术表述，存为 `results_text` 变量

**关键 API**：
- `from statsmodels.stats.weightstats import ttest_ind` -- t 检验
- `from scipy.stats import chi2_contingency` -- 卡方检验
- Cohen's d = (mean1 - mean2) / pooled_std


In [ ]:
# TODO 4：撰写 Results（用 statsmodels 跑统计检验）
# 提示：
#   np.random.seed(42)
#   control_eff = np.random.normal(8.2, 2.5, 200)   # 人工组效率
#   treatment_eff = np.random.normal(32.5, 8.0, 200)  # Agent组效率
#   t_stat, p_val, df = ttest_ind(treatment_eff, control_eff)
#   cohen_d = (treatment_eff.mean() - control_eff.mean()) / np.sqrt(
#       ((len(control_eff)-1)*control_eff.std()**2 + (len(treatment_eff)-1)*treatment_eff.std()**2)
#       / (len(control_eff) + len(treatment_eff) - 2))
#   卡方检验: chi2, p, dof, expected = sp_stats.chi2_contingency([[clicked_ctrl, not_ctrl], [clicked_trt, not_trt]])
# 要求：跑 t检验+Cohen's d+卡方检验，将结果写成 APA 格式存为 results_text

# ===== 你的代码 =====
np.random.seed(42)
# 生成 A/B 测试数据
control_efficiency = None       # TODO: 对照组效率（均值~8.2, std~2.5, N=200）
treatment_efficiency = None     # TODO: 实验组效率（均值~32.5, std~8.0, N=200）

# t 检验
t_stat = None    # TODO: t 统计量
p_val = None     # TODO: p 值
df_val = None    # TODO: 自由度

# Cohen's d
cohen_d = None   # TODO: 效应量

# 卡方检验（CTR: 对照组 2.1% vs 实验组 2.8%, N=10000每组）
chi2_stat = None  # TODO: 卡方统计量
chi2_p = None     # TODO: 卡方 p 值

results_text = None  # TODO: 将统计结果写成 APA 格式学术表述
# ====================

print(f"t 检验: t({df_val}) = {t_stat:.2f}, p = {p_val:.4e}")
print(f"Cohen's d = {cohen_d:.2f}")
print(f"卡方检验: χ² = {chi2_stat:.2f}, p = {chi2_p:.4f}")
print("\n" + "="*60)
print(results_text)


## 5. IMRaD 详解：Discussion（论文的灵魂）

Discussion 是论文的"灵魂"--它展示了你对研究的深度理解。

**Discussion 六要素**：
1. 主要发现解读：不只是重复 Results，要解释"为什么"
2. 理论贡献：扩展了什么理论、提出了什么框架
3. 实践启示：对企业有什么 actionable 的建议
4. 局限性：诚实承认（样本/时间/评估/技术局限）
5. 未来研究方向：2-3 个方向
6. 研究伦理声明：IRB/知情同意/数据脱敏

> 详见独立教材 § 3.6.5。


## TODO 5：撰写 Discussion

**要求**：
1. 主要发现解读：Agent 在效率上显著优于人工（d=2.34），但23.5%的人工修改率表明无法完全替代
2. 理论贡献：扩展 DSR 在 AI 系统设计中的应用
3. 实践启示：Agent 应定位为"辅助工具"而非"替代方案"
4. 局限性：样本局限/时间局限/LLM-as-a-judge 评估偏差/技术时效性
5. 未来研究方向：跨企业验证/长期影响/个性化能力
6. 研究伦理声明

将撰写的 Discussion 存为 `discussion_text` 变量（多行字符串）。


In [ ]:
# TODO 5：撰写 Discussion
# 提示：发现解读/理论贡献/实践启示/局限性/未来方向/伦理声明
# 要求：基于 Results 的统计结果撰写 Discussion

# ===== 你的代码 =====
discussion_text = None  # TODO: 撰写完整的 Discussion（多行字符串）
# ====================

print(discussion_text)


## 6. APA 第 7 版引用规范

APA（American Psychological Association）格式是社会科学领域最常用的引用格式。

**正文引用**：
- 单作者：(Smith, 2025)
- 两作者：(Smith & Jones, 2025)
- 三作者及以上：(Smith et al., 2025)
- 直接引用：(Smith, 2025, p. 15)
- 机构作者：(NIST, 2024)

**参考文献列表格式**：
- 期刊论文：Author, A. B., & Author, C. D. (Year). Title. *Journal Name*, Vol(Issue), pages. https://doi.org/xxx
- 会议论文：Author, A. B. et al. (Year). Title. In *Conference Name*.
- arXiv 预印本：Author, A. B. (Year). Title. arXiv preprint arXiv:XXXX.XXXXX.
- 书籍：Author, A. B. (Year). *Title* (ed.). Publisher.

> 详见独立教材 § 3.6.6。


## TODO 6：生成 APA 第 7 版参考文献列表

**要求**：
基于以下真实引用，生成 APA 第 7 版格式的参考文献列表：

1. Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2022). ReAct: Synergizing reasoning and acting in language models. arXiv preprint arXiv:2210.03629.
2. Zheng, L., Chiang, W. L., Sheng, Y., et al. (2023). Judging LLM-as-a-judge with MT-Bench and Chatbot Arena. arXiv preprint arXiv:2306.05685.
3. Peffers, K., Tuunanen, T., Rothenberger, M. A., & Chatterjee, S. (2007). A design science research methodology for information systems research. *Journal of Management Information Systems*, 24(3), 45-78.
4. Creswell, J. W., & Plano Clark, V. L. (2018). *Designing and conducting mixed methods research* (3rd ed.). SAGE Publications.
5. Braun, V., & Clarke, V. (2006). Using thematic analysis in psychology. *Qualitative Research in Psychology*, 3(2), 77-101.
6. Cohen, J. (1988). *Statistical power analysis for the behavioral sciences* (2nd ed.). Lawrence Erlbaum Associates.

将参考文献列表存为 `references_text` 变量（多行字符串）。


In [ ]:
# TODO 6：生成 APA 第 7 版参考文献列表
# 提示：按 APA 第 7 版格式排列（作者. (年份). 标题. *期刊*, 卷(期), 页码. DOI）
# 要求：生成6条真实引用的 APA 格式参考文献

# ===== 你的代码 =====
references_text = None  # TODO: 生成 APA 第 7 版参考文献列表
# ====================

print(references_text)


## 7. 反思与前沿

### 反思问题
1. 你的 Introduction 是否清晰陈述了研究问题和贡献？能否让非专业人士理解你为什么做这个研究？
2. 你的 Methods 是否足够详细，别人能复现你的研究？哪些细节最容易遗漏？
3. Results 中的统计检验结果是否用 APA 格式正确报告了？效应量大小意味着什么？
4. Discussion 中的局限性是否诚实？有没有"报喜不报忧"的倾向？

### 2026 前沿：LLM-as-a-judge 评估写作质量
在 IMRaD 论文写作中，**LLM-as-a-judge**（NeurIPS 2023, arXiv 2306.05685）可用于：
- **写作质量自检**：让 LLM 扮演"论文审稿人"，按预设 criteria（IMRaD 结构完整性/逻辑连贯性/学术表述规范性）自动评分
- **引用格式校验**：用 LLM 检查参考文献是否符合 APA 第 7 版格式
- **逻辑链审查**：让 LLM 追踪 Introduction -> Methods -> Results -> Discussion 的逻辑一致性

**注意**：LLM-as-a-judge 是辅助工具，有自身偏差（偏好长答案/位置偏差/自我偏好）。它对应因果阶梯 L1（对文本的关联分析），不能替代真实同行评审（L2 干预：修改后重新提交）。定位为"投稿前自检工具"。

参考 [arXiv 2306.05685](https://arxiv.org/abs/2306.05685)（NeurIPS 2023, LLM-as-a-judge）+ [arxiv Python 包](https://github.com/lukasschwab/arxiv.py)。
